# TensorEE quickstart: map water with a trained U-Net, inside Earth Engine

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelEOS/tensor-ee/blob/main/notebooks/01_quickstart.ipynb)

- **What it does:** runs a trained water-mapping U-Net on one Sentinel-1 chip
  (224 × 224 pixels at 10 m) at a place you pick, and shows the water map.
- **How:** Earth Engine computes the whole network itself. Nothing is downloaded, and the
  weights are public Earth Engine assets, so you need no model file.
- **You need:** an Earth Engine account and a Cloud project
  [registered for Earth Engine](https://developers.google.com/earth-engine/guides/access).
- **To run:** enter your project in step 2, then **Runtime → Run all**.
- **Cost:** the run uses your project's Earth Engine quota (EECU time). That is free on a
  noncommercial project; a commercial project pays its plan's rate. The example's export used
  about 65 EECU-seconds (64.9 recorded), plus a few small requests.

In [ ]:
# @title 1. Install (about a minute)
# The model's Earth Engine computation is in the repository, not on PyPI, so install from it.
!git clone --quiet --depth 1 https://github.com/noelEOS/tensor-ee.git
%pip install --quiet ./tensor-ee folium

import sys
from pathlib import Path

REPOSITORY = next(folder.resolve() for folder in (Path("tensor-ee"), Path("."), Path(".."))
                  if (folder / "demos" / "code_editor" / "segwater.js").is_file())
sys.path.insert(0, str(REPOSITORY))
print("TensorEE is installed.")

In [ ]:
# @title 2. Your project and place
# @markdown - **PROJECT**: your Cloud project ID, as in the Code Editor. Nothing else needs editing.
# @markdown - The default place is Sun Moon Lake, Taiwan. The date range picks a Sentinel-1 scene.
PROJECT = ""  # @param {type:"string"}
LONGITUDE = 120.915  # @param {type:"number"}
LATITUDE = 23.862  # @param {type:"number"}
START_DATE = "2024-01-01"  # @param {type:"date"}
END_DATE = "2024-01-08"  # @param {type:"date"}

import time

ASSET_NAME = "tensor_ee_quickstart_" + time.strftime("%Y%m%d_%H%M%S")   # a new asset each run


class Stop(Exception):
    """Stops the notebook with a one-line message instead of a traceback."""

    def _render_traceback_(self):
        return [f"Stopped: {self}"]


TASKS_URL = "https://code.earthengine.google.com/tasks"


def wait_for(task, poll=20):
    """Wait for an export, with a short status line every minute; stop if it fails."""
    started, polls = time.time(), 0
    while (state := task.status()["state"]) not in ("COMPLETED", "FAILED", "CANCELLED"):
        time.sleep(poll)
        polls += 1
        if polls % 3 == 0:
            minutes = (time.time() - started) / 60
            print(f"  still running, {minutes:.0f} min ({state.lower()})...", flush=True)
    if state != "COMPLETED":
        raise Stop(f"the export ended {state}: {task.status().get('error_message', '')} (see {TASKS_URL})")


if not PROJECT.strip():
    raise Stop("enter your Cloud project ID in PROJECT above, then run all cells again.")
print(f"Settings saved: project {PROJECT}, point ({LONGITUDE}, {LATITUDE}), "
      f"scenes from {START_DATE} to {END_DATE}.")

In [ ]:
# @title 3. Sign in
# @markdown The first time, follow the link with the Google account you use for Earth Engine.
import ee

ee.Authenticate()
ee.Initialize(project=PROJECT)
print(f"Earth Engine is ready — signed in to {PROJECT}.")

In [ ]:
# @title 4. Find the chip and a Sentinel-1 scene (2 small requests)
# @markdown A 224 × 224 chip of 10 m pixels in the local UTM zone, centred on your point,
# @markdown and the first scene in the date range that covers all of it.
from tensor_ee import Grid

CHIP, PIXEL = 224, 10
zone = min(60, max(1, int((LONGITUDE + 180) // 6) + 1))
crs = f"EPSG:{326 if LATITUDE >= 0 else 327}{zone:02d}"
x, y = ee.Geometry.Point([LONGITUDE, LATITUDE]).transform(crs, 0.01).coordinates().getInfo()
half = CHIP * PIXEL // 2
place = {"crs": crs, "x": round((x - half) / PIXEL) * PIXEL, "y": round((y + half) / PIXEL) * PIXEL}
grid = Grid(crs, (PIXEL, 0, place["x"], 0, -PIXEL, place["y"]), CHIP, CHIP)

scenes = (ee.ImageCollection("COPERNICUS/S1_GRD")
          .filterBounds(grid.geometry()).filterDate(START_DATE, END_DATE)
          .filter(ee.Filter.eq("instrumentMode", "IW"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
          .filter(ee.Filter.eq("resolution_meters", 10))
          .sort("system:time_start").limit(10))


def coverage(image):
    valid = image.select(["VV", "VH"]).mask().reduce(ee.Reducer.min()).gt(0).unmask(0, False)
    share = valid.rename("valid").reduceRegion(
        reducer=ee.Reducer.mean(), geometry=grid.geometry(), crs=grid.crs,
        crsTransform=list(grid.transform), maxPixels=CHIP * CHIP, tileScale=2).get("valid")
    return ee.Feature(None, {"id": ee.String("COPERNICUS/S1_GRD/").cat(image.get("system:index")),
                             "share": share})


found = scenes.map(coverage).reduceColumns(ee.Reducer.toList(2), ["id", "share"]).get("list").getInfo()
complete = [scene for scene, share in found if share >= 0.999]
if not complete:
    raise Stop(f"no Sentinel-1 scene covers the whole chip between {START_DATE} and {END_DATE}; "
               "widen the dates or move the point.")
scene_id = complete[0]
print(f"Found the chip ({CHIP} × {CHIP} px in {crs}) and a Sentinel-1 scene that covers it: {scene_id}")

In [ ]:
# @title 5. Set up the model for your chip
# @markdown Loads the trained U-Net, stored in the repository as Earth Engine operations, and points
# @markdown it at your chip and scene. Nothing runs yet.
import ee
from demos.code_editor import build as segwater_module
from tensor_ee import Grid

sys.setrecursionlimit(max(sys.getrecursionlimit(), 20000))   # the network is deeply nested
TEMPLATE, METADATA = segwater_module.parse_module(
    (REPOSITORY / "demos" / "code_editor" / "segwater.js").read_text())
BANDS = list(segwater_module.OUTPUT_BANDS)

grid = Grid(place["crs"], (10, 0, place["x"], 0, -10, place["y"]), 224, 224)
text = segwater_module.substitute(TEMPLATE, grid.crs, scene_id,
                                  (place["x"], place["y"]), (place["x"], place["y"]))
chip = ee.Image(ee.deserializer.fromJSON(text))
prediction = (ee.ImageCollection.fromImages([chip]).mosaic().select(BANDS)
              .setDefaultProjection(grid.projection()).clip(grid.geometry())
              .updateMask(grid.mask()).toFloat())
print(f"Ready: the trained U-Net is set up for your chip and scene "
      f"({len(METADATA['weightAssets'])} public weight tables). Nothing has run yet; step 6 runs it.")

In [ ]:
# @title 6. Run the model (an export of about 65 EECU-seconds, a few minutes)
# @markdown Earth Engine computes the chip in a batch export and saves it as a new image asset.
# @markdown Follow it in the [Tasks tab](https://code.earthengine.google.com/tasks) too.
asset_id = f"projects/{PROJECT}/assets/{ASSET_NAME}"
task = ee.batch.Export.image.toAsset(
    image=prediction.set({"source_scene": scene_id}), description="tensor_ee_quickstart",
    assetId=asset_id, region=grid.geometry(), crs=grid.crs, crsTransform=list(grid.transform),
    maxPixels=224 * 224 * len(BANDS), pyramidingPolicy={".default": "sample"})
task.start()
print("Export started; it usually takes a few minutes. Follow it here or at", TASKS_URL)
wait_for(task)
print("Done: the water map is saved as", asset_id)

In [ ]:
# @title 7. See the radar image and the water map
# @markdown Waits for step 6's export if it is still running. Layers: radar, water probability,
# @markdown water mask (1 where the probability is at least 0.5); switch them top right.
import folium

if "task" in globals() and task.status()["state"] != "COMPLETED":
    print("Waiting for step 6's export to finish...")
    wait_for(task)
elif "task" not in globals():
    try:
        ee.data.getAsset(f"projects/{PROJECT}/assets/{ASSET_NAME}")
    except ee.EEException:
        raise Stop("run step 6 first: it creates the water map this step shows.") from None
asset_id = f"projects/{PROJECT}/assets/{ASSET_NAME}"

def layer(image, visualization, name, show=True):
    url = ee.Image(image).getMapId(visualization)["tile_fetcher"].url_format
    return folium.raster_layers.TileLayer(
        tiles=url, name=name, overlay=True, show=show,
        attr="Google Earth Engine; contains modified Copernicus Sentinel data")

result = ee.Image(asset_id)
centre_lon, centre_lat = grid.geometry().centroid(1).transform("EPSG:4326", 1).coordinates().getInfo()
ESRI = "https://server.arcgisonline.com/ArcGIS/rest/services/{}/MapServer/tile/{{z}}/{{y}}/{{x}}"
water_map = folium.Map(location=[centre_lat, centre_lon], zoom_start=14, tiles=None)
folium.raster_layers.TileLayer(tiles=ESRI.format("Canvas/World_Light_Gray_Base"),
                               attr="Tiles &copy; Esri", name="Esri Light Gray").add_to(water_map)
folium.raster_layers.TileLayer(
    tiles=ESRI.format("World_Imagery"),
    attr="Tiles &copy; Esri &mdash; Source: Esri, Maxar, Earthstar Geographics, and the GIS User Community",
    name="Esri World Imagery").add_to(water_map)
layer(ee.Image(scene_id).select("VV"), {"min": -25, "max": 0}, "Sentinel-1 VV (dB)").add_to(water_map)
layer(result.select("water_probability"), {"min": 0, "max": 1, "palette": ["ffffff", "08306b"]},
      "Water probability").add_to(water_map)
layer(result.select("water_mask").selfMask(), {"palette": ["00bfff"]}, "Water mask",
      show=False).add_to(water_map)
folium.LayerControl().add_to(water_map)
water_map

## Good to know

- **Checked, not guaranteed accurate.** At Sun Moon Lake, this computation matched the trained
  model in PyTorch to 1.35e-5 in the logits, with identical water masks
  ([validation](https://github.com/noelEOS/tensor-ee/blob/main/docs/validation.md)). That is
  agreement, not accuracy: the map is a prediction, not ground truth
  ([model notes](https://github.com/noelEOS/tensor-ee/blob/main/docs/model.md)).
- **Why an export:** the whole network in one interactive request exceeds Earth Engine's memory
  limit, so it runs as a batch task. The map then reads the saved asset, which stays in your
  project until you delete it (Code Editor, **Assets** tab).
- **Next:** [is my model supported?](https://github.com/noelEOS/tensor-ee/blob/main/notebooks/02_is_my_model_supported.ipynb)
  and [bring your own CNN](https://github.com/noelEOS/tensor-ee/blob/main/notebooks/03_bring_your_own_cnn.ipynb);
  [start here](https://github.com/noelEOS/tensor-ee/blob/main/docs/start-here.md) for the background.